# Predicting deprivation

## Idea

We are going to try to use satellite data to predict deprivation and show that, indeed, satellites are better than old data, and embeddings are better than manual features.

### Outline

1. Data prep
    - Landsat: obtain cloud-free images/composites for London extent for 2019 and 2025
    - Embeddings: download our [data product](https://data.imago.ac.uk/datasets/google-satellite-embedding-v1-london-lsoas-2020-2024)
    - Geometries: use those from the embeddings files
1. Setup
    - Train/test split
    - Performance ($R^2$, $RMSE$)
1. Model fitting (2019)
    - m1/ `IMD19 ~ Manual features`
    - m2/ `IMD19 ~ Embeddings`
1. _Who is better at predicting contemporaneous IMD?_ --> Model comparison (m1 Vs. m2), winner?
1. _What about _future_ IMD?_
    - Build predictions for 2025 with m1 ($\hat{m1}$) and m2 ($\hat{m2}$)
    - Compare $\hat{m1}$, $\hat{m2}$, _and_ $IMD_{19}$

## Data

- LSOAs for London (2020 def)
- IMD
    - 2019
    - 2025
- Landsat to build features
    - 2019
    - 2025
- LSOA Embeddings
    - 2019
    - 2024

Resources:

- IMD [2019](https://www.gov.uk/government/statistics/english-indices-of-deprivation-2019) and [2025](https://deprivation.communities.gov.uk/download-all)
- [GeoVisualisation lab](https://gdsl-ul.github.io/wma/labs/w07_pixelsToPatterns.html)
- [London embeddings](https://data.imago.ac.uk/datasets/google-satellite-embedding-v1-london-lsoas-2020-2024)

In [1]:
import pandas
import geopandas
import numpy as np
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import GroupKFold

results = []

# Load data
imd = pandas.read_csv('imd.csv', index_col='LSOA21CD')
emb = geopandas.read_file('uk_lsoa_london_embeds_2020.geojson').set_index('LSOA21CD')

emb_cols = [c for c in emb.columns if c.endswith('_mean')]
db = imd[['imd19_score', 'led19_score', 'grid_id']].join(emb[emb_cols])


ERROR 1: PROJ: proj_create_from_database: Open of /opt/conda/envs/gds/share/proj failed


## CV-based training on IMD'19

In [2]:
gkf = GroupKFold(n_splits=5)

for target in ['imd19_score', 'led19_score']:
    sub = db.dropna(subset=[target] + emb_cols)
    X = sub[emb_cols].values
    y = sub[target].values
    groups = sub['grid_id'].values

    fold_r2, fold_rmse = [], []
    for tr_idx, te_idx in gkf.split(X, y, groups):
        model = HistGradientBoostingRegressor(random_state=42)
        model.fit(X[tr_idx], y[tr_idx])
        y_pred = model.predict(X[te_idx])
        fold_r2.append(r2_score(y[te_idx], y_pred))
        fold_rmse.append(root_mean_squared_error(y[te_idx], y_pred))

    print(
        f"{target}  —  "
        f"R²: {np.mean(fold_r2):.3f} ± {np.std(fold_r2):.3f}  |  "
        f"RMSE: {np.mean(fold_rmse):.3f} ± {np.std(fold_rmse):.3f}"
    )
    idx = 'IMD' if target.startswith('imd') else 'LED'
    results.append({
        'Features': 'Embeddings 2020',
        'Target': target,
        'Setting': f'{idx} 2019 (CV)',
        'R²': np.mean(fold_r2),
        'R² std': np.std(fold_r2),
        'RMSE': np.mean(fold_rmse),
        'RMSE std': np.std(fold_rmse),
        'Spearman ρ': float('nan'),
    })


imd19_score  —  R²: 0.104 ± 0.077  |  RMSE: 10.145 ± 0.381
led19_score  —  R²: 0.304 ± 0.081  |  RMSE: 8.454 ± 0.989


## CV-based training on IMD'25 (rank targets)

In [3]:
# 2025 IMD — embeddings from 2024, targets are ranks
emb24 = geopandas.read_file('uk_lsoa_london_embeds_2024.geojson').set_index('LSOA21CD')
emb_cols24 = [c for c in emb24.columns if c.endswith('_mean')]
db24 = imd[['imd25_rank', 'led25_rank', 'grid_id']].join(emb24[emb_cols24])

for target in ['imd25_rank', 'led25_rank']:
    sub = db24.dropna(subset=[target] + emb_cols24)
    X = sub[emb_cols24].values
    y = sub[target].values
    groups = sub['grid_id'].values

    fold_r2, fold_rmse, fold_rho = [], [], []
    for tr_idx, te_idx in gkf.split(X, y, groups):
        model = HistGradientBoostingRegressor(random_state=42)
        model.fit(X[tr_idx], y[tr_idx])
        y_pred = model.predict(X[te_idx])
        fold_r2.append(r2_score(y[te_idx], y_pred))
        fold_rmse.append(root_mean_squared_error(y[te_idx], y_pred))
        fold_rho.append(spearmanr(y[te_idx], y_pred)[0])

    print(
        f"{target}  —  "
        f"R²: {np.mean(fold_r2):.3f} ± {np.std(fold_r2):.3f}  |  "
        f"RMSE: {np.mean(fold_rmse):.3f} ± {np.std(fold_rmse):.3f}  |  "
        f"ρ: {np.mean(fold_rho):.3f} ± {np.std(fold_rho):.3f}"
    )
    idx = 'IMD' if target.startswith('imd') else 'LED'
    results.append({
        'Features': 'Embeddings 2024',
        'Target': target,
        'Setting': f'{idx} 2025 (CV)',
        'R²': np.mean(fold_r2),
        'R² std': np.std(fold_r2),
        'RMSE': np.mean(fold_rmse),
        'RMSE std': np.std(fold_rmse),
        'Spearman ρ': np.mean(fold_rho),
    })


imd25_rank  —  R²: 0.093 ± 0.173  |  RMSE: 7896.497 ± 306.573  |  ρ: 0.422 ± 0.080
led25_rank  —  R²: 0.359 ± 0.085  |  RMSE: 4773.625 ± 494.099  |  ρ: 0.619 ± 0.063


## Census'21 for IMD'25

We now test whether 2021 Census socioeconomic indicators—qualifications, self-reported health, population density, and lone-family households—can predict 2025 deprivation ranks. This serves as the manual-feature baseline (m1 in the outline) against which embeddings are compared.

In [4]:
soc = pandas.read_csv('Socioeconomic.csv', index_col='LSOA21CD').drop(columns='Unnamed: 0')
soc_cols = [
    'Percent no qualifications 16 and over',
    'Percent bad and very band health',
    'Population density per km',
    'Percent lone family household',
]
db_soc = imd[['imd25_rank', 'led25_rank', 'grid_id']].join(soc[soc_cols])

for target in ['imd25_rank', 'led25_rank']:
    sub = db_soc.dropna(subset=[target] + soc_cols)
    X = sub[soc_cols].values
    y = sub[target].values
    groups = sub['grid_id'].values

    fold_r2, fold_rmse, fold_rho = [], [], []
    for tr_idx, te_idx in gkf.split(X, y, groups):
        model = HistGradientBoostingRegressor(random_state=42)
        model.fit(X[tr_idx], y[tr_idx])
        y_pred = model.predict(X[te_idx])
        fold_r2.append(r2_score(y[te_idx], y_pred))
        fold_rmse.append(root_mean_squared_error(y[te_idx], y_pred))
        fold_rho.append(spearmanr(y[te_idx], y_pred)[0])

    print(
        f"{target} (census)  —  "
        f"R²: {np.mean(fold_r2):.3f} ± {np.std(fold_r2):.3f}  |  "
        f"RMSE: {np.mean(fold_rmse):.3f} ± {np.std(fold_rmse):.3f}  |  "
        f"ρ: {np.mean(fold_rho):.3f} ± {np.std(fold_rho):.3f}"
    )
    idx = 'IMD' if target.startswith('imd') else 'LED'
    results.append({
        'Features': 'Census 2021',
        'Target': target,
        'Setting': f'{idx} 2025 (CV)',
        'R²': np.mean(fold_r2),
        'R² std': np.std(fold_r2),
        'RMSE': np.mean(fold_rmse),
        'RMSE std': np.std(fold_rmse),
        'Spearman ρ': np.mean(fold_rho),
    })


imd25_rank (census)  —  R²: 0.510 ± 0.098  |  RMSE: 5806.917 ± 481.694  |  ρ: 0.739 ± 0.059
led25_rank (census)  —  R²: -0.053 ± 0.142  |  RMSE: 6132.907 ± 743.771  |  ρ: 0.254 ± 0.152


## Census'21 + embeddings'24 for IMD'25 (CV)

We combine the 2021 Census features with the 2024 satellite embeddings to test whether the two sources of information are complementary when predicting 2025 deprivation ranks.

In [5]:
combined_cols = soc_cols + emb_cols24
db_combined = imd[['imd25_rank', 'led25_rank', 'grid_id']].join(soc[soc_cols]).join(emb24[emb_cols24])

for target in ['imd25_rank', 'led25_rank']:
    sub = db_combined.dropna(subset=[target] + combined_cols)
    X = sub[combined_cols].values
    y = sub[target].values
    groups = sub['grid_id'].values

    fold_r2, fold_rmse, fold_rho = [], [], []
    for tr_idx, te_idx in gkf.split(X, y, groups):
        model = HistGradientBoostingRegressor(random_state=42)
        model.fit(X[tr_idx], y[tr_idx])
        y_pred = model.predict(X[te_idx])
        fold_r2.append(r2_score(y[te_idx], y_pred))
        fold_rmse.append(root_mean_squared_error(y[te_idx], y_pred))
        fold_rho.append(spearmanr(y[te_idx], y_pred)[0])

    print(
        f"{target} (census + embeddings)  —  "
        f"R²: {np.mean(fold_r2):.3f} ± {np.std(fold_r2):.3f}  |  "
        f"RMSE: {np.mean(fold_rmse):.3f} ± {np.std(fold_rmse):.3f}  |  "
        f"ρ: {np.mean(fold_rho):.3f} ± {np.std(fold_rho):.3f}"
    )
    idx = 'IMD' if target.startswith('imd') else 'LED'
    results.append({
        'Features': 'Census 2021 + Embeddings 2024',
        'Target': target,
        'Setting': f'{idx} 2025 (CV)',
        'R²': np.mean(fold_r2),
        'R² std': np.std(fold_r2),
        'RMSE': np.mean(fold_rmse),
        'RMSE std': np.std(fold_rmse),
        'Spearman ρ': np.mean(fold_rho),
    })


imd25_rank (census + embeddings)  —  R²: 0.737 ± 0.061  |  RMSE: 4236.788 ± 314.753  |  ρ: 0.860 ± 0.040
led25_rank (census + embeddings)  —  R²: 0.382 ± 0.086  |  RMSE: 4693.020 ± 536.411  |  ρ: 0.639 ± 0.066


## Embeddings'19 on IMD'25 ranks

In [6]:
from scipy.stats import spearmanr

# Fit final models on all 2019 data (no CV — we're forecasting, not evaluating 2019)
db19_full = imd[['imd19_score', 'led19_score']].join(emb[emb_cols]).dropna()
db25_full = imd[['imd25_rank', 'led25_rank']].join(emb24[emb_cols24]).dropna()

for target19, target25 in [('imd19_score', 'imd25_rank'), ('led19_score', 'led25_rank')]:
    model = HistGradientBoostingRegressor(random_state=42)
    model.fit(db19_full[emb_cols].values, db19_full[target19].values)

    # Predict 2025 scores; invert to ranks so high score → rank 1 (most deprived),
    # matching imd25_rank convention where 1 = most deprived
    common = db25_full.index.intersection(db19_full.index)
    sub25 = db25_full.loc[common]
    pred_ranks = (
        pandas.Series(model.predict(sub25[emb_cols24].values), index=sub25.index)
        .rank(ascending=False)
    )
    actual_ranks = sub25[target25].rank()

    rho, pval = spearmanr(actual_ranks, pred_ranks)
    r2 = r2_score(actual_ranks, pred_ranks)
    print(
        f"{target19} → {target25}  —  "
        f"Spearman ρ: {rho:.3f} (p={pval:.2e})  |  R²: {r2:.3f}"
    )
    results.append({
        'Features': 'Embeddings 2020 → 2024 (forecast)',
        'Target': target25,
        'Setting': 'Forecast',
        'R²': r2,
        'R² std': float('nan'),
        'RMSE': float('nan'),
        'RMSE std': float('nan'),
        'Spearman ρ': rho,
    })


imd19_score → imd25_rank  —  Spearman ρ: 0.564 (p=0.00e+00)  |  R²: 0.128
led19_score → led25_rank  —  Spearman ρ: 0.637 (p=0.00e+00)  |  R²: 0.275


## Baseline: IMD'19 for IMD'25

As a naive baseline, we use the 2019 IMD score itself—without any model—to forecast the 2025 rank. This captures pure rank persistence: how much does knowing where an area stood in 2019 already tell you about where it will stand in 2025?

In [7]:
# Baseline: use 2019 IMD directly as predictor for 2025 (no model — rank persistence)
for score19, rank25 in [('imd19_score', 'imd25_rank'), ('led19_score', 'led25_rank')]:
    sub = imd.dropna(subset=[score19, rank25])

    # Higher 2019 score = more deprived → rank 1 (matches imd25_rank where 1 = most deprived)
    pred_rank = sub[score19].rank(ascending=False)

    # imd25_rank is a national rank; convert to London-only rank
    actual_rank = sub[rank25].rank()

    rho, pval = spearmanr(actual_rank, pred_rank)
    r2 = r2_score(actual_rank, pred_rank)
    print(
        f"{score19} → {rank25}  —  "
        f"Spearman ρ: {rho:.3f} (p={pval:.2e})  |  R²: {r2:.3f}"
    )
    results.append({
        'Features': 'IMD 2019 (baseline)',
        'Target': rank25,
        'Setting': 'Baseline',
        'R²': r2,
        'R² std': float('nan'),
        'RMSE': float('nan'),
        'RMSE std': float('nan'),
        'Spearman ρ': rho,
    })


imd19_score → imd25_rank  —  Spearman ρ: 0.930 (p=0.00e+00)  |  R²: 0.860
led19_score → led25_rank  —  Spearman ρ: 0.728 (p=0.00e+00)  |  R²: 0.456


## Summary: model comparison

The table below consolidates all results across features, targets, and evaluation settings. CV models report cross-validated mean ± std; forecast and baseline models report point estimates only. RMSE is omitted where not applicable (rank-comparison experiments).


In [8]:
from IPython.display import display

tbl = pandas.DataFrame(results)

def _fmt(mean, std=None):
    if pandas.isna(mean):
        return ''
    if std is not None and not pandas.isna(std):
        return f'{mean:.3f} ± {std:.3f}'
    return f'{mean:.3f}'

table_styles = [
    {'selector': 'caption',
     'props': [('font-size', '1.1em'), ('font-weight', 'bold'), ('margin-bottom', '0.5em')]},
    {'selector': 'th',
     'props': [('background-color', '#e8e8e8'), ('font-weight', 'bold'), ('text-align', 'left')]},
    {'selector': 'td',
     'props': [('text-align', 'left'), ('padding', '4px 12px')]},
    {'selector': 'tr:nth-child(even)',
     'props': [('background-color', '#f7f7f7')]},
]

for index_label, caption in [('imd', 'IMD'), ('led', 'LED')]:
    sub = tbl[tbl['Target'].str.startswith(index_label)].copy()
    summary = pandas.DataFrame({
        'Features':   sub['Features'],
        'Setting':    sub['Setting'],
        'R²':         [_fmt(r, s) for r, s in zip(sub['R²'], sub['R² std'])],
        'RMSE':       [_fmt(r, s) for r, s in zip(sub['RMSE'], sub['RMSE std'])],
        'Spearman ρ': [_fmt(v) for v in sub['Spearman ρ']],
    }).reset_index(drop=True)
    display(
        summary.style
        .hide(axis='index')
        .set_caption(f'{caption} — model comparison')
        .set_table_styles(table_styles)
    )


Features,Setting,R²,RMSE,Spearman ρ
Embeddings 2020,IMD 2019 (CV),0.104 ± 0.077,10.145 ± 0.381,
Embeddings 2024,IMD 2025 (CV),0.093 ± 0.173,7896.497 ± 306.573,0.422
Census 2021,IMD 2025 (CV),0.510 ± 0.098,5806.917 ± 481.694,0.739
Census 2021 + Embeddings 2024,IMD 2025 (CV),0.737 ± 0.061,4236.788 ± 314.753,0.860
Embeddings 2020 → 2024 (forecast),Forecast,0.128,,0.564
IMD 2019 (baseline),Baseline,0.860,,0.930


Features,Setting,R²,RMSE,Spearman ρ
Embeddings 2020,LED 2019 (CV),0.304 ± 0.081,8.454 ± 0.989,
Embeddings 2024,LED 2025 (CV),0.359 ± 0.085,4773.625 ± 494.099,0.619
Census 2021,LED 2025 (CV),-0.053 ± 0.142,6132.907 ± 743.771,0.254
Census 2021 + Embeddings 2024,LED 2025 (CV),0.382 ± 0.086,4693.020 ± 536.411,0.639
Embeddings 2020 → 2024 (forecast),Forecast,0.275,,0.637
IMD 2019 (baseline),Baseline,0.456,,0.728
